# t-SNE & UMAP

**Companion lesson:** https://ml-viz.vercel.app/courses/pca-dimensionality/02-t-sne-and-umap

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## PCA vs t-SNE

PCA finds linear projections that maximize variance. t-SNE preserves local neighborhood structure.

In [ ]:
from sklearn.decomposition import PCA

digits = load_digits()
X, y = digits.data, digits.target

X_pca = PCA(n_components=2).fit_transform(X)
X_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', s=5, alpha=0.7)
axes[0].set_title('PCA — Linear Projection', color='white', fontsize=12)

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='tab10', s=5, alpha=0.7)
axes[1].set_title('t-SNE — Preserves Local Structure', color='white', fontsize=12)

for ax in axes:
    ax.axis('off')
plt.suptitle('MNIST Digits: PCA vs t-SNE', color='white', fontsize=13)
plt.tight_layout()
plt.show()

## Effect of perplexity in t-SNE

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, perp in zip(axes, [5, 30, 100]):
    X_embedded = TSNE(n_components=2, perplexity=perp, random_state=42).fit_transform(X)
    ax.scatter(X_embedded[:, 0], X_embedded[:, 1], c=y, cmap='tab10', s=5, alpha=0.7)
    ax.set_title(f'Perplexity = {perp}', color='white', fontsize=11)
    ax.axis('off')
plt.suptitle('t-SNE: Perplexity Controls Local vs Global Structure', color='white', fontsize=13)
plt.tight_layout()
plt.show()

## What perplexity actually does: tuning sigma

Perplexity is $2^{H(P_i)}$ where $H$ is the Shannon entropy (bits) of point $i$'s conditional neighbor distribution $p_{j|i}\propto e^{-d_{ij}^2/2\sigma_i^2}$. t-SNE binary-searches each $\sigma_i$ so the perplexity matches the target — i.e. so each point has a chosen *effective number of neighbors*. Below we reproduce that search by hand and confirm the identities from the lesson.

In [ ]:
import numpy as np

def perplexity_of(p):
    """Perplexity = 2^H, H = Shannon entropy (bits) of distribution p."""
    p = p[p > 0]
    H = -np.sum(p * np.log2(p))
    return 2.0 ** H

# Identities from the lesson:
print('uniform over 4 neighbors -> perplexity =', perplexity_of(np.full(4, 0.25)))   # 4.0
print('peaked (0.97,0.01,0.01,0.01) -> perplexity =',
      round(perplexity_of(np.array([0.97, 0.01, 0.01, 0.01])), 3))                    # ~1.18

# Conditional p_{j|i} from squared distances with bandwidth sigma
def conditional_p(d2, sigma):
    w = np.exp(-d2 / (2 * sigma**2))
    return w / w.sum()

def sigma_for_perplexity(d2, target, lo=1e-3, hi=1e3, iters=60):
    """Binary-search sigma so perplexity(p_{j|i}) == target (t-SNE's exact procedure)."""
    for _ in range(iters):
        mid = (lo + hi) / 2
        perp = perplexity_of(conditional_p(d2, mid))
        if perp < target:   # too peaked -> widen sigma
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

# One point with 10 neighbors at increasing distances
rng = np.random.default_rng(0)
d2 = np.sort(rng.uniform(0.5, 30, size=10))
for target in [2, 5, 8]:
    s = sigma_for_perplexity(d2, target)
    p = conditional_p(d2, s)
    print(f'target perp {target}: sigma={s:.3f}, achieved perp={perplexity_of(p):.3f}, '
          f'top-3 p={np.round(np.sort(p)[::-1][:3], 3)}')
print('\nHigher target perplexity -> larger sigma -> probability spread over MORE neighbors.')


## UMAP: faster, preserves more global structure

UMAP often beats t-SNE on speed and keeps more of the global layout. Key knobs: `n_neighbors` (local vs global) and `min_dist` (cluster tightness).

In [ ]:
from sklearn.datasets import load_digits
X, y = load_digits(return_X_y=True)
try:
    import umap
    emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(X)
    plt.scatter(emb[:, 0], emb[:, 1], c=y, cmap='Spectral', s=6)
    plt.title('UMAP of digits'); plt.colorbar(label='digit'); plt.show()
except ImportError:
    print('pip install umap-learn to run this cell')

## Key takeaways

- **t-SNE** and **UMAP** are non-linear methods for **visualizing** high-dim data in 2-D.
- They preserve **local** neighborhoods; distances/sizes between clusters aren't literal.
- t-SNE's **perplexity** and UMAP's **n_neighbors** balance local vs global structure.
- Use PCA to denoise first; never use these embeddings as features for a downstream model.